# 05-三极管与 MOS 管


## 1. 本篇学习范围

本篇对应课程视频 **35—39**：三极管入门、NPN 三极管仿真、PNP 三极管仿真、三极管的放大特性、MOS 管和三极管。

这一篇把第 04 篇的“控制负载”继续展开：用一个较小的控制信号，控制另一条回路中的电流，并理解开关与放大的区别。

| 主题 | 学完后应能回答的问题 |
|---|---|
| 半导体与 PN 结 | 二极管为什么具有单向导电特性？ |
| 三极管的端子与类型 | B、C、E 是什么？NPN 和 PNP 怎样判断导通？ |
| 开关工作 | 怎样用三极管控制 LED、蜂鸣器或继电器线圈？ |
| 放大工作 | 小信号怎样影响较大的电流？放大的能量来自哪里？ |
| MOS 管 | G、D、S 是什么？怎样判断 GPIO 能否驱动它？ |
| 器件比较 | 什么时候选三极管，什么时候考虑 MOS 管？ |

> 本篇以课程主题和配套讲义为主。计算例、GPIO 接口说明、MOS 管选型及仿真步骤属于学习补充；其中的数值是练习条件，不是原视频中电路参数的转录。


## 2. 先理解半导体与二极管

课程从硅、掺杂和二极管引出三极管。这里先抓住三个概念：

- **半导体**：导电能力可以通过掺杂等方式改变，硅是常见的半导体材料。
- **N 型半导体**：多数载流子是电子。
- **P 型半导体**：多数载流子是空穴，可以把空穴理解为电子缺位所表现出的正电荷运动。

P 型和 N 型材料形成 **PN 结**，这是理解二极管和双极型三极管的基础。

### 二极管的基本行为

| 状态 | 常见 PN 结二极管的表现 |
|---|---|
| 正向偏置 | 正向电流随电压上升而明显增加 |
| 反向偏置且未击穿 | 只有较小的漏电流，近似截止 |
| 反向电压达到击穿条件 | 可能出现较大的反向电流，是否允许取决于器件类型和限流条件 |

普通硅二极管在一定电流下，正向压降常用约 `0.6～0.7 V` 估算；它随器件、电流和温度变化。LED、肖特基二极管等不能直接套用这个数值。

**必要澄清：**讲义把 P、N 分别与 positive、negative 联系起来帮助记忆。这里的 P/N 指多数载流子的类型，不表示整块 P 型材料整体带正电、N 型材料整体带负电；均匀掺杂的材料整体通常仍近似电中性。


## 3. 三极管的三个端子：B、C、E

本篇所说的“三极管”主要指 **双极型晶体管 BJT**。它有 NPN、PNP 两种常见类型。

| 标记 | 英文 | 中文 | 初步理解 |
|---|---|---|---|
| B | Base | 基极 | 施加控制信号的端子 |
| C | Collector | 集电极 | 主电流回路的一端 |
| E | Emitter | 发射极 | 主电流回路的另一端，也是判断基极电压的参考端 |

三极管符号中的箭头画在 **发射极** 上：NPN 箭头向外，PNP 箭头向内。箭头表示相应 PN 结正向导通时的常规电流方向。

### 看符号与看实物是两件事

原理图用 B/C/E 明确电气连接；实物三个引脚的排列，需要查具体型号的数据手册。

- 相同的 TO-92 外形，可能有不同的引脚顺序。
- 同系列名称也要核对厂商、完整型号和封装。
- 数据手册中的顶视图、底视图、正面朝向不能混用。

> 讲义中的 TO-92 图片用于认识封装和端子，不能把图中的引脚顺序当作所有三极管的通用规则。


## 4. NPN 与 PNP：导通要看基极相对发射极的电压

定义：

```text
V_BE = V_B - V_E
V_CE = V_C - V_E
```

| 项目 | NPN | PNP |
|---|---|---|
| 使基极—发射极结正向偏置 | B 比 E 高约一个结压降 | B 比 E 低约一个结压降 |
| 硅管常用估算 | V_BE 约 +0.6～+0.8 V | V_BE 约 -0.6～-0.8 V |
| 正常工作时主回路的常规电流方向 | C → E | E → C |
| 基极电流方向 | 流入 B | 从 B 流出 |
| 常见开关接法 | 发射极接地，控制负载的低端 | 发射极接正电源，控制负载的高端 |

讲义中的“NPN 高电平导通、PNP 低电平导通”适合对应的常见接法。更可靠的判断方法始终是：**先找 E，再比较 B 和 E 的电压。**

例如，PNP 的 E 接 `5 V`：

- B 接近 `5 V`，基极—发射极结没有明显正向偏置，通常截止；
- B 被经电阻拉低，有足够的基极驱动，就可以导通。

这不能简单替换成“只要是 GPIO 高电平就关断”。如果 GPIO 的高电平只有 `3.3 V`，它与 `5 V` 发射极之间仍存在明显电压差。

> 这些结压降是粗略估算，不是严格的数字开关阈值。基极—发射极电压影响结电流，外部电路必须同时提供合适的基极电流。


## 5. 截止、放大、饱和：三种常用工作状态

把三极管用作开关与用作放大器时，工作目标不同。

| 工作状态 | 简化表现 | 常见用途 |
|---|---|---|
| 截止 | 基极驱动不足，集电极电流很小，主回路近似断开 | 开关的“关” |
| 正向放大区 | 基极—发射极结正偏、基极—集电极结反偏，小信号变化可以控制主回路电流变化 | 信号放大 |
| 饱和 | 两个结都正偏，主回路压降较小，电流主要受电源和负载限制 | 开关的“开” |

在正向放大区，可以用电流的大小作近似计算：

```text
I_C ≈ β × I_B
I_E = I_C + I_B
```

`β` 是共射极电流放大系数，数据手册中常见 `h_FE`。它会随型号、样品、电流、温度和工作电压变化。

### 饱和时不能继续套用 β 倍关系

假设 `β=100`、`I_B=0.2 mA`，计算得到 `I_C≈20 mA`。如果电源与负载只允许约 `5 mA`，三极管并不会强行产生 `20 mA`；它会进入饱和，负载电流由外部回路决定。

硅三极管的饱和压降 `V_CE(sat)` 有时可以用约 `0.1～0.3 V` 估算，实际必须看具体电流和基极驱动条件下的数据手册。

**记忆方式：**开关希望在截止和饱和之间切换；线性放大需要合适的静态工作点，使信号变化时尽量保持在放大区。


## 6. NPN 低端开关：控制 LED 和其他小负载

低端开关把器件放在负载与地之间。

```text
+5 V ── R_LED ── LED ── C
                       │
                    NPN 三极管
                       │
GND ────────────────── E

GPIO ── R_B ── B
                 │
               R_BE（可选）
                 │
GND ─────────────┘
```

图中 LED 正极朝向电源；`R_LED` 是 LED 限流电阻，`R_B` 是基极限流电阻，`R_BE` 是基极到发射极的下拉电阻。

| 控制状态 | 常见结果 |
|---|---|
| GPIO 为低电平 | B 接近 E，三极管截止，LED 熄灭 |
| GPIO 为高电平且基极驱动足够 | 三极管进入饱和，负载回路形成，LED 点亮 |
| GPIO 处于高阻态 | 下拉电阻帮助保持关断，避免基极悬空 |

### 接线的三个检查点

1. **必须有基极限流。**基极—发射极结类似正向二极管，不能把普通 GPIO 直接接 B。
2. **负载有自己的限流或额定条件。**基极电阻不能代替 LED 的限流电阻。
3. **非隔离驱动需要共地。**GPIO 所属控制板的地与负载电源的地连接，才有明确的控制电压参考。

如果负载换成继电器线圈，需要按第 04 篇的方法加续流保护：普通低端直流线圈的二极管阴极接电源正端、阳极接 C；正常通电时反偏，关断时为线圈电流提供回路。


## 7. PNP 高端开关：从电源侧控制负载

高端开关把器件放在正电源与负载之间。

```text
+5 V ───────────── E
                   │
                PNP 三极管
                   │
                   C ── R_LED ── LED ── GND

控制端 ── R_B ── B
                 │
               R_BE（可选）
                 │
+5 V ────────────┘
```

这里的 `R_BE` 是基极到发射极的上拉电阻。

- 控制端接近 `+5 V`：B 与 E 电位接近，通常关断。
- 控制端经电阻被拉低：基极电流从 E 经 B 流向控制端，三极管可导通。
- 导通后主电流从 E 流向 C，再通过 LED 和电阻回到地。

### 3.3 V 单片机控制 5 V 高端负载时要做接口匹配

不能把“PNP 低电平导通”直接变成“任意 GPIO 都能直接连接”。直接连接可能导致关断不充分，或使 GPIO 承受超出允许范围的电压、电流。

常见补充方案是：PNP 的 B 用电阻上拉到 E，再用一个 NPN 或专用驱动器经限流电阻把 B 拉低。前级 NPN 导通时 PNP 导通；前级关断时，PNP 的 B 被拉回 E。

> 本节的直接控制图用于解释原理，前提是控制端电压范围与发射极电源相匹配。实际接口还应核对控制芯片引脚的允许范围和上电默认状态。


## 8. 开关电路怎样估算电阻和 GPIO 驱动能力

这一节是与课程主题相关的计算练习。不要只查一个典型 `h_FE`，就认为开关一定能充分导通。

### 练习一：3.3 V 控制信号驱动 5 V LED 回路

先假设：控制输出高电平 `V_OH=3.3 V`，LED 正向压降 `2.0 V`，三极管饱和压降 `0.2 V`，期望 LED 电流约 `10 mA`。

```text
R_LED = (V_CC - V_LED - V_CE(sat)) / I_LED
      = (5 - 2.0 - 0.2) / 0.010
      = 280 Ω
```

选择常用值 `330 Ω` 后：

```text
I_LED ≈ 2.8 / 330 ≈ 8.5 mA
P_R ≈ I² × R ≈ 0.024 W
```

再用**强迫电流放大系数** `β_forced=I_C/I_B=10` 做初步估算。这里的 10 是练习假设，应结合数据手册的饱和测试条件选择。

```text
按 10 mA 负载预留：I_B ≈ 10 mA / 10 = 1 mA
假设 V_BE≈0.8 V：R_B ≈ (3.3 - 0.8) / 0.001 = 2.5 kΩ
取 R_B=2.2 kΩ：I_B ≈ 2.5 / 2200 ≈ 1.14 mA
```

实际的 `V_OH` 可能随着 GPIO 输出电流增加而降低，`V_BE`、`V_CE(sat)` 也会变化；因此计算只是初选，最终要核对最不利条件和实测结果。

### 练习二：负载电流增大后，GPIO 还能承担吗？

如果直流继电器线圈需要 `70 mA`，仍假设 `β_forced=10`：

```text
I_B ≈ 7 mA
R_B ≈ (3.3 - 0.8) / 0.007 ≈ 357 Ω
```

取 `330 Ω`，估算基极电流约 `7.6 mA`。这个数值未必适合所用 GPIO；必须检查单脚驱动能力、对应 `V_OH`、端口及芯片总电流限制。必要时改用合适的 MOS 管或驱动器。

**选电阻前的顺序：**先确定负载电流，再确定器件及驱动条件，最后计算电阻并检查 GPIO，而不是先随意放一个电阻。


## 9. NPN 与 PNP 仿真：看三个量，不只看“亮不亮”

课程视频 36、37 分别演示 NPN、PNP。下面给出便于复习的 CircuitJS 练习条件；不同版本菜单和模型名称可能略有差异。

### NPN 仿真

搭建：`5 V → 1 kΩ 电阻 → C`，E 接地；B 经 `10 kΩ` 电阻连接一个可在 `0～3.3 V` 间变化的直流控制源，两路电源共地。

观察：基极电流 `I_B`、集电极电流 `I_C`、集电极—发射极电压 `V_CE`。

| 操作 | 预期观察 |
|---|---|
| 控制电压接近 0 V | 接近截止，I_C 很小，C 被电阻拉到接近 5 V |
| 缓慢提高控制电压 | 基极开始有电流，I_C 增大，V_CE 降低 |
| 继续增加基极驱动 | 进入饱和后，I_C 不再按 β 倍无限增大 |

若饱和压降约 `0.2 V`，电阻允许的集电极电流约为：

```text
I_C ≈ (5 - 0.2) / 1000 = 4.8 mA
```

### PNP 仿真

把 E 接 `+5 V`，C 经 `1 kΩ` 电阻接地；B 经 `10 kΩ` 电阻连接一个可在 `0～5 V` 间变化的控制源，负端共地。

- 控制端约 `5 V`：B 接近 E，接近截止。
- 逐步降低控制电压：`V_E - V_B` 增大，基极电流从 B 流出，负载电流增大。
- 足够导通后：C 接近正电源，E 与 C 之间的压降减小。

PNP 的电流和 `V_CE` 在软件中可能显示为负值，要结合软件箭头和电压参考端解释。比较电流大小时，可以先看绝对值，再核对实际方向。

### 记录模板

| 类型 | 控制电压 | V_BE | I_B | I_C | V_CE | 判断状态 |
|---|---|---|---|---|---|---|
| NPN | 低、中、高各记录一次 | 实测 | 实测 | 实测 | 实测 | 截止/放大/饱和 |
| PNP | 高、中、低各记录一次 | 实测 | 实测 | 实测 | 实测 | 截止/放大/饱和 |

> “预期观察”是根据电路关系作出的预测；本篇没有把未实际运行的仿真写成实验测量结果。实物还需考虑模型未完整呈现的参数离散、温升和接口限制。


## 10. 三极管的放大特性：放大来自电源供能

三极管可以用较小的基极电流变化，控制较大的集电极电流变化。输出能量来自电源，输入信号负责控制能量的变化。

在合适的工作点附近，作近似分析：

```text
ΔI_C ≈ β × ΔI_B
```

### 电流变化怎样变成电压变化

一个简化的共射极放大回路：

```text
+5 V ── R_C ── C ── 输出 V_out
                │
               NPN
                │
GND ─────────── E

B 端接“直流偏置 + 小交流信号”
```

由于集电极电阻上有压降：

```text
V_out ≈ V_CC - I_C × R_C
```

所以基极驱动增强、集电极电流增大时，集电极输出电压反而降低，形成共射极电路的**反相**效果。

假设一个已正确偏置的理想电路：`R_C=2 kΩ`、静态 `I_C=1 mA`、E 近似为 `0 V`，则静态输出约为 `3 V`。再假设 `β=100`，基极电流增加 `2 μA`：

```text
ΔI_C ≈ 100 × 2 μA = 0.2 mA
ΔV_out ≈ -2 kΩ × 0.2 mA = -0.4 V
```

这只是工作点附近的近似。电流放大系数 `β` 不等于整个电路的电压放大倍数；实际电压增益还取决于电阻、偏置和负载等条件。

### 为什么要有静态工作点

若三极管原本截止，交流信号的一部分可能无法导通；若原本已经饱和，继续增加输入也无法按比例改变输出，波形就会削顶或失真。

实际放大器要设计偏置和稳定措施，上面的图不是省略偏置后即可照接的完整电路。普通正弦小信号直接接一个没有偏置的基极，通常不能得到完整的线性放大波形。


## 11. 课程应用案例：非接触式电笔中的多级放大

配套讲义第 68、69 页用非接触式电笔说明微弱信号与多级三极管放大。图中的 J1 是一小块金属感应片，Q1、Q2、Q3 逐级连接，最终影响 LED 回路。

从原理上理解这条链：

```text
交流导线附近的变化电场
        ↓ 微小电容耦合
金属感应片 J1 上出现微弱信号
        ↓
Q1 → Q2 → Q3 逐级放大/增强驱动
        ↓
由电池供能的 LED 回路产生可观察的变化
```

讲义图中的连接可按**多级达林顿式连接**理解：前一级发射极连接后一级基极，集电极连在一起。为了理解放大趋势，有时把总电流增益粗略看成各级增益相乘；这不适用于所有电流、频率或饱和状态，也会放大漏电和干扰的影响。

这里要分清两件事：

- 感应片提供的是微弱控制信号，LED 的主要能量来自电池。
- 图示是在解释放大链；实际 LED 回路仍需适当限流，不能把示意图中省略的限流措施当成通用接法。

**应用边界：**非接触检测会受距离、姿态、屏蔽和环境耦合影响；指示灯没有反应不能证明线路已经断电。本篇只理解信号放大原理，具体电笔原理图与 PCB 留到课程后续案例展开。


## 12. MOS 管的三个端子与控制方式

MOS 管是场效应晶体管的一类，常见端子是：

| 标记 | 英文 | 中文 |
|---|---|---|
| G | Gate | 栅极 |
| D | Drain | 漏极 |
| S | Source | 源极 |

课程讲义用“水龙头控制水流”比喻晶体管控制电流，第 66 页图中也出现了栅极、源极、漏极和 MOS 结构。这个比喻帮助理解控制关系；**BJT 的 B/C/E 与 MOS 管的 G/D/S 不能混为同一组端子。**

常见的增强型 MOS 管分为 N 沟道、P 沟道。判断控制条件时看：

```text
V_GS = V_G - V_S
```

| 类型 | 基本导通方向 | 常见接法 |
|---|---|---|
| N 沟道增强型 MOS 管 | G 相对 S 足够为正时形成导电沟道 | S 接地，做低端开关 |
| P 沟道增强型 MOS 管 | G 相对 S 足够为负时形成导电沟道 | S 接正电源，做高端开关 |

### 栅极输入电流小，但切换时仍需充放电

MOS 管的栅极与沟道之间有绝缘层，静态栅极电流通常很小；不过栅极有电容，开通、关断时驱动源必须给它充放电。

因此“电压控制”不能理解成“任何情况下都不需要驱动电流”。在频繁切换、大栅极电荷或高频 PWM 场景中，驱动能力会影响切换速度、损耗和发热。


## 13. N 沟道 MOS 管低端开关与 P 沟道高端开关

### N 沟道低端开关

```text
+V ── 负载 ── D
               │
           N 沟道 MOS 管
               │
GND ────────── S

GPIO ── R_G ── G
                │
              R_GS
                │
GND ───────────┘
```

- S 接地时，`3.3 V` 栅极高电平对应约 `V_GS=3.3 V`。
- `R_GS` 是栅极到源极的下拉电阻，帮助在 GPIO 高阻或上电期间关断。
- 小串联栅极电阻 `R_G` 可限制瞬间充放电电流、抑制振铃，具体阻值需要结合驱动器和开关速度选择。
- 控制板与负载电源需要共地；负载电流应通过功率回路回流，避免细小的信号线路承受它。
- 继电器线圈等感性负载仍需续流保护，二极管阴极接正电源、阳极接 D。

### P 沟道高端开关

S 接正电源，D 接负载，负载另一端接地；G 接近 S 时关断，把 G 拉低到相对 S 足够负时导通。

例如 S 接 `5 V`，G 拉到 `0 V` 时 `V_GS=-5 V`；G 只抬到 `3.3 V` 时仍有 `V_GS=-1.7 V`，不能据此保证关断。

实际可以让栅极通过电阻上拉到 S，再用合适的电平转换或驱动器拉低，同时检查最大允许 `|V_GS|`。不能让 GPIO 直接承受高于其允许范围的电源电压。

### 体二极管不是隔离触点

常见离散功率 MOS 管内部存在体二极管。因此一个 MOS 管关断后，并不等价于继电器触点在两个方向都断开。D/S 不应随意对调，双向阻断通常需要额外电路。


## 14. MOS 管选型：阈值电压不等于充分导通电压

这是从“认识器件”进入实际驱动时最重要的检查之一。

`V_GS(th)` 是在数据手册规定的较小电流等测试条件下，开始形成导通沟道的阈值参数。它不能证明 MOS 管已经可以低损耗地带动目标负载。

例如：某管的 `V_GS(th)` 标为 `1～2 V`，但低导通电阻只在 `V_GS=10 V` 条件下保证，不能直接认定 `3.3 V GPIO` 足以让它充分导通。

| 参数 | 需要核对的内容 |
|---|---|
| 引脚及封装 | G/D/S 排列、散热焊盘与电气连接 |
| V_DS 额定值 | 是否覆盖负载供电和可能的瞬态电压 |
| 最大允许 V_GS | 栅极电压范围是否安全 |
| R_DS(on) | 在实际可提供的栅压下，是否有保证值；热态下会怎样变化 |
| I_D 与安全工作区 SOA | 电流额定值依赖哪些温度、散热和工作条件 |
| 栅极电荷 Q_g | 驱动电路能否满足切换速度要求 |
| 热阻与散热条件 | 封装、PCB 铜皮、环境温度是否允许目标功耗 |

### 导通损耗的简单估算

假设器件在实际栅压和温度条件下的导通电阻为 `0.05 Ω`：

```text
P_cond ≈ I² × R_DS(on)
1 A 时：P_cond ≈ 1² × 0.05 = 0.05 W
3 A 时：P_cond ≈ 3² × 0.05 = 0.45 W
```

这只估算导通期间的损耗，切换损耗等还没有加入。温度升高后导通电阻可能增大；数据手册中的大电流额定值也不代表小封装可以在任意 PCB 上长期承受该电流。

> 选给 3.3 V GPIO 使用的器件，应查找在对应低栅压条件下有明确性能保证的型号，并核对输出高电平的最不利情况。“逻辑电平 MOS 管”这个名称本身不能代替参数表。


## 15. 三极管与 MOS 管怎样比较

视频 39 将两类器件放在一起比较。对初学者，先比较控制条件和负载需求。

| 维度 | BJT 三极管 | MOS 管 |
|---|---|---|
| 端子 | B、C、E | G、D、S |
| 初步控制模型 | 基极需要持续的驱动电流，V_BE 同时影响结电流 | 栅源电压控制沟道，切换时栅极需要充放电 |
| 开关设计的重点 | 基极电流是否足够、V_CE(sat)、GPIO 能力 | 实际 V_GS 下的 R_DS(on)、栅极驱动和热条件 |
| 常用低端类型 | NPN | N 沟道增强型 |
| 常用高端类型 | PNP | P 沟道增强型 |
| 导通损耗初估 | P ≈ V_CE(sat) × I_C，基极驱动也有损耗 | P ≈ I² × R_DS(on)，还要考虑切换损耗 |
| 典型入门场景 | 小电流 LED、简单信号放大、适当的负载驱动 | 较大电流负载、低损耗开关、PWM 驱动 |
| 容易忽视的点 | β 的离散性、饱和状态、实物脚序 | 阈值与充分导通的区别、体二极管、栅极悬空 |

一个简单的选择过程：

```text
负载需要多大电流、什么供电电压？
        ↓
控制端能提供多少电压和电流？
        ↓
器件在这些条件下能否满足导通压降、功耗和额定值？
        ↓
检查接线、默认状态、感性负载保护，再进行低压验证
```

MOS 管和 BJT 都能做开关，也都能用于放大。不能简单归纳成“MOS 只做开关、三极管只做放大”，更不能认为某类器件在所有情况下都更合适。


## 16. 常见误区与排查顺序

| 现象或说法 | 应检查什么 |
|---|---|
| GPIO 高电平，负载却不动作 | 是否共地？脚序是否正确？BJT 的基极电流或 MOS 的实际 V_GS 是否足够？ |
| LED 很快损坏 | LED 是否有独立限流电阻？电流是否超过允许值？ |
| 三极管明显发热 | 是否未充分饱和、负载过大或超出安全工作区？测 I_C 和 V_CE |
| MOS 管带载时发热 | 该栅压下 R_DS(on) 是否足够小？切换是否过慢？散热是否足够？ |
| 上电时负载误动作 | 基极/栅极是否悬空？下拉或上拉是否合适？GPIO 默认状态是什么？ |
| PNP/P 沟道器件关不掉 | 关断时 B/G 是否真正接近 E/S，而不是只达到 GPIO 的高电平？ |
| “β=100，所以任何时候 I_C 都是 100 I_B” | 先判断是否在放大区，再检查电源和负载限制 |
| “MOS 栅极不流电，所以不需要驱动能力” | 静态漏电与切换时充放电电流要分开考虑 |

### 用万用表做低压检查

先断电核对型号、封装和线路；通电后按第 03 篇的并联测压方法测端子电压。

- NPN：测 `V_B`、`V_E`、`V_C`，计算 `V_BE` 和 `V_CE`。
- PNP：仍按同样定义计算，别把负号误认成故障；也可看 `V_E-V_B` 与 `V_E-V_C` 的大小。
- MOS：测 G 与 S 两端的电压差，不能只测 G 对地就断定驱动正常。
- 若需要测电流，按串联测流方法接表，或通过已知电阻上的压降估算。

在脱离电路、断电的条件下，二极管档可以辅助检查 BJT 的两个 PN 结；但测得两个结并不能单独可靠区分所有器件的 C/E，也不能代替数据手册和完整性能检查。


## 17. 一页复习表

| 知识点 | 关键结论 |
|---|---|
| P/N 型半导体 | 分别以空穴/电子为多数载流子，材料整体不因此带净正/负电 |
| BJT 端子 | B 基极、C 集电极、E 发射极，脚序查具体型号 |
| NPN 导通 | B 相对 E 为正，且有足够的基极驱动 |
| PNP 导通 | B 相对 E 为负，且有合适的基极电流路径 |
| 工作状态 | 开关用截止/饱和，线性放大需要合适的工作点 |
| 电流放大 | 放大区近似 I_C≈βI_B，β 不是固定常数 |
| BJT 开关 | 基极要限流；饱和后电流主要受负载和供电限制 |
| 放大能量 | 来自电源，输入信号负责控制变化 |
| MOS 端子 | G 栅极、D 漏极、S 源极 |
| MOS 控制 | 看 V_GS，切换时栅极需要充放电 |
| MOS 选型 | 看实际栅压下的 R_DS(on)，不能只看 V_GS(th) |
| 高端开关 | 关断控制电位需要接近 E/S，注意与 GPIO 电压匹配 |
| 感性负载 | 用 BJT 或 MOS 驱动都要考虑关断续流和瞬态保护 |

### 本篇核心公式

```text
V_BE = V_B - V_E
V_CE = V_C - V_E
V_GS = V_G - V_S

放大区：I_C ≈ β × I_B
电流大小：I_E = I_C + I_B
开关初估：I_B ≈ I_C / β_forced
NPN 基极电阻初估：R_B ≈ (V_OH - V_BE) / I_B

共射极输出：V_out ≈ V_CC - I_C × R_C
BJT 饱和导通损耗初估：P ≈ V_CE(sat) × I_C
MOS 导通损耗初估：P ≈ I² × R_DS(on)
```

上述公式各有适用条件；尤其不能把放大区关系套到饱和区，也不能把典型参数当作全工况保证值。


## 18. 自测题

1. NPN 的 E 接地，B 电位比 E 高约 `0.7 V`，是否就能保证它可以驱动任意电流的负载？
2. PNP 的 E 接 `5 V`，B 分别接近 `5 V` 和被经电阻拉到更低电位时，状态有何区别？
3. 三极管做开关与做线性放大时，希望工作在哪些状态？
4. `β=100`、`I_B=0.1 mA`，是否总有 `I_C=10 mA`？
5. `5 V` 回路中 LED 压降为 `2 V`，NPN 饱和压降为 `0.2 V`，串联电阻为 `330 Ω`，LED 电流约是多少？
6. 假设 GPIO 高电平为 `3.3 V`、基极压降为 `0.8 V`，期望基极电流为 `1 mA`，基极电阻初估是多少？
7. 共射极电路中 I_C 增大时，集电极输出电压为什么降低？输出能量来自哪里？
8. 某 MOS 管阈值电压为 `1.5 V`，是否可以据此认定 `3.3 V GPIO` 能让它低损耗地驱动 `2 A` 负载？
9. MOS 管的导通电阻为 `0.1 Ω`，流过 `2 A` 时导通损耗约是多少？
10. 为什么 GPIO 高阻时要考虑基极/栅极的上下拉？MOS 管关断后能否当作双向断开的继电器触点？
11. 为什么非接触式电笔没有亮灯不能直接证明线路已经断电？

### 参考答案

1. 不能。这个压降只说明结可能已明显正向导通，还要检查基极电流、器件额定值、负载及散热等条件。
2. B 接近 E 时通常截止；B 经限流电阻拉低、有足够基极驱动时可导通，主电流从 E 流向 C。
3. 开关希望在截止和饱和之间切换；线性放大希望信号变化范围内保持在正向放大区。
4. 不总成立。它只是放大区的近似，还受 β 实际值、电源及负载限制；饱和时不能继续套用。
5. `I≈(5-2-0.2)/330≈8.5 mA`。
6. `R_B≈(3.3-0.8)/0.001=2.5 kΩ`，常用值选择后还需核对 GPIO 与器件参数。
7. `V_out≈V_CC-I_C R_C`，电流增加使电阻压降增加、输出电位降低；放大所需能量来自电源。
8. 不能。要核对实际栅压下保证的 R_DS(on)、允许电流、热条件和驱动能力。
9. `P≈2²×0.1=0.4 W`，这里只计算导通损耗。
10. 上下拉帮助建立确定的默认状态；常见离散功率 MOS 管有体二极管，单管关断通常不能保证双向阻断。
11. 检测可能受屏蔽、距离、姿态和环境耦合影响，缺少指示不等于证明无电。


## 19. 与原课程资料的对应关系

- 视频 `35_三极管入门.mp4`：对应半导体、二极管、B/C/E 端子、NPN/PNP 与基本控制关系。
- 视频 `36_npn三极管仿真.mp4`：对应 NPN 接线、基极控制与电流、电压的仿真观察。
- 视频 `37_pnp三极管的仿真.mp4`：对应 PNP 极性、导通条件及与 NPN 的对照。
- 视频 `38_三极管的放大特性.mp4`：对应电流放大、工作状态及多级放大的理解。
- 视频 `39_mos管和三极管.mp4`：对应 MOS 管与 BJT 的端子、控制方式和应用比较。
- 配套 `ppt/模拟电路.pdf`：已在线核对第 **63—69 页**的半导体、二极管、三极管定义、工作原理、两种三极管和非接触式电笔放大案例；第 70 页开始进入 NE555。
- 原资料：[百度网盘课程分享](https://pan.baidu.com/s/1AZwoEpfx_2anpJ8p9ofD-Q?pwd=9988)，提取码 `9988`。

本篇依据已核对的课程目录和配套 PDF 在线预览整理，**视频未逐字转录**。电阻计算、仿真练习参数、静态工作点说明、GPIO 接口匹配与 MOS 数据手册检查为学习补充。讲义中“电压控制”“高/低电平导通”等概括已补充适用条件；半导体电中性、端子区分和感应案例的解释用于避免入门误解。

下一篇：**06-NE555 定时器与振荡电路**，进入 NE555 的引脚、比较器与内部逻辑、外围电路、方波产生和电子琴原理。
